# Exercise 1: Neural Networks from Scratch
## Building a Neural Network Using Only NumPy

---

### Learning Objectives
By the end of this exercise you will:
- Understand the forward pass through a neural network
- Derive and implement backpropagation manually
- Visualize how a network learns a non-linear decision boundary
- Compare the effect of different activation functions

### The Challenge
You will train a neural network **without any deep learning framework** (no PyTorch, no TensorFlow) to classify a **spiral dataset** — a problem that is impossible for a linear classifier but solvable with a 2-layer network.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.datasets import make_moons, make_circles
from IPython.display import HTML

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries loaded!')

## Part 1: Generate the Spiral Dataset

The spiral dataset has 3 classes arranged in interlocking spirals. **No linear classifier can separate these.** It's a classic benchmark for testing whether a network can learn non-linear features.

In [ ]:
def generate_spiral(N=100, K=3):
    """Generate N points per class for K classes arranged in spirals."""
    X = np.zeros((N * K, 2))
    y = np.zeros(N * K, dtype='uint8')
    for j in range(K):
        ix = range(N * j, N * (j + 1))
        r = np.linspace(0.0, 1, N)
        t = np.linspace(j * 4, (j + 1) * 4, N) + np.random.randn(N) * 0.2
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    return X, y

X, y = generate_spiral(N=100, K=3)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#2ecc71', '#3498db']
for cls in range(3):
    mask = y == cls
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[cls], label=f'Class {cls}', s=30, edgecolors='white', linewidths=0.5)
axes[0].set_title('Spiral Dataset (3 Classes)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].set_aspect('equal')

# Compare: can a linear classifier separate this?
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000).fit(X, y)
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
Z = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
for cls in range(3):
    mask = y == cls
    axes[1].scatter(X[mask, 0], X[mask, 1], c=colors[cls], label=f'Class {cls}', s=30, edgecolors='white', linewidths=0.5)
axes[1].set_title(f'Logistic Regression: {accuracy_score(y, lr.predict(X)):.1%} accuracy (should fail!)', fontsize=12)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()
print(f"Linear classifier accuracy: {accuracy_score(y, lr.predict(X)):.1%} — compare with ~33% random chance")

## Part 2: Build the Neural Network

Our network architecture:
```
Input (2) → Hidden Layer (64, ReLU) → Hidden Layer (32, ReLU) → Output (3, Softmax)
```

### Forward Pass
$$z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$$
$$a^{[l]} = g(z^{[l]})$$

### Loss (Cross-Entropy)
$$L = -\frac{1}{N}\sum_{i}\sum_{k} y_{ik} \log(\hat{y}_{ik})$$

In [ ]:
# ============================================================
#  Activation functions — complete the missing implementations
# ============================================================

def relu(z):
    return np.maximum(0, z)

def relu_backward(dA, z):
    return dA * (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_backward(dA, z):
    s = sigmoid(z)
    return dA * s * (1 - s)

def tanh_backward(dA, z):
    return dA * (1 - np.tanh(z) ** 2)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Visualize activation functions
z = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, fn, dfn) in zip(axes, [
    ('Sigmoid', sigmoid, lambda x: sigmoid(x) * (1 - sigmoid(x))),
    ('ReLU', relu, lambda x: (x > 0).astype(float)),
    ('Tanh', np.tanh, lambda x: 1 - np.tanh(x)**2),
]):
    ax.plot(z, fn(z), 'b-', linewidth=2.5, label='f(z)')
    ax.plot(z, dfn(z), 'r--', linewidth=2, label="f'(z)")
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_ylim(-1.5, 2)

plt.suptitle('Activation Functions and Their Derivatives', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_dims, activation='relu', lr=0.01):
        """
        layer_dims: list of ints, e.g. [2, 64, 32, 3]
        """
        self.lr = lr
        self.activation = activation
        self.params = {}
        self.cache = {}
        self.loss_history = []
        
        # He initialization for ReLU, Xavier for sigmoid/tanh
        for l in range(1, len(layer_dims)):
            fan_in = layer_dims[l - 1]
            scale = np.sqrt(2.0 / fan_in) if activation == 'relu' else np.sqrt(1.0 / fan_in)
            self.params[f'W{l}'] = np.random.randn(layer_dims[l - 1], layer_dims[l]) * scale
            self.params[f'b{l}'] = np.zeros((1, layer_dims[l]))
        
        self.L = len(layer_dims) - 1

    def forward(self, X):
        self.cache['A0'] = X
        A = X
        for l in range(1, self.L + 1):
            W, b = self.params[f'W{l}'], self.params[f'b{l}']
            Z = A @ W + b
            self.cache[f'Z{l}'] = Z
            if l == self.L:
                A = softmax(Z)
            elif self.activation == 'relu':
                A = relu(Z)
            elif self.activation == 'sigmoid':
                A = sigmoid(Z)
            else:
                A = np.tanh(Z)
            self.cache[f'A{l}'] = A
        return A

    def compute_loss(self, y_pred, y_true):
        N = y_true.shape[0]
        # One-hot encode
        one_hot = np.zeros_like(y_pred)
        one_hot[np.arange(N), y_true] = 1
        loss = -np.sum(one_hot * np.log(y_pred + 1e-15)) / N
        return loss

    def backward(self, y_true):
        N = y_true.shape[0]
        grads = {}
        
        # Gradient of softmax + cross-entropy combined (dL/dZ_last)
        one_hot = np.zeros_like(self.cache[f'A{self.L}'])
        one_hot[np.arange(N), y_true] = 1
        dA = (self.cache[f'A{self.L}'] - one_hot) / N
        
        for l in reversed(range(1, self.L + 1)):
            A_prev = self.cache[f'A{l-1}']
            Z = self.cache[f'Z{l}']
            W = self.params[f'W{l}']
            
            if l < self.L:
                if self.activation == 'relu':
                    dA = relu_backward(dA, Z)
                elif self.activation == 'sigmoid':
                    dA = sigmoid_backward(dA, Z)
                else:
                    dA = tanh_backward(dA, Z)
            
            grads[f'dW{l}'] = A_prev.T @ dA
            grads[f'db{l}'] = np.sum(dA, axis=0, keepdims=True)
            dA = dA @ W.T
        
        return grads

    def update(self, grads):
        for l in range(1, self.L + 1):
            self.params[f'W{l}'] -= self.lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= self.lr * grads[f'db{l}']

    def train(self, X, y, epochs=5000, verbose=True):
        self.boundary_snapshots = []
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y_pred, y)
            grads = self.backward(y)
            self.update(grads)
            self.loss_history.append(loss)
            
            if epoch % 500 == 0:
                self.boundary_snapshots.append((epoch, self.predict(X)))
                if verbose:
                    acc = np.mean(np.argmax(y_pred, axis=1) == y)
                    print(f'Epoch {epoch:5d} | Loss: {loss:.4f} | Accuracy: {acc:.1%}')

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

print('NeuralNetwork class defined!')

In [ ]:
# Train the network
nn = NeuralNetwork(layer_dims=[2, 64, 32, 3], activation='relu', lr=0.5)
nn.train(X, y, epochs=5000)

In [ ]:
def plot_decision_boundary(model, X, y, title=''):
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.35, cmap='RdYlBu', levels=[-0.5, 0.5, 1.5, 2.5])
    plt.contour(xx, yy, Z, colors='white', linewidths=0.5, levels=[0.5, 1.5])
    for cls in range(3):
        mask = y == cls
        plt.scatter(X[mask, 0], X[mask, 1], c=colors[cls], s=30, 
                    edgecolors='white', linewidths=0.5, label=f'Class {cls}')
    acc = np.mean(model.predict(X) == y)
    plt.title(f'{title}\nAccuracy: {acc:.1%}', fontweight='bold')
    plt.legend(loc='upper right')
    plt.axis('equal')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
plot_decision_boundary(nn, X, y, 'Our NumPy Network (ReLU)')

axes[1].plot(nn.loss_history, color='#e74c3c', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[1].set_title('Training Loss Curve', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## Part 3: The Vanishing Gradient Problem

Compare training deep networks with **Sigmoid** vs **ReLU**. Watch what happens to gradient magnitudes as the network gets deeper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results = {}
for activation in ['sigmoid', 'relu']:
    model = NeuralNetwork([2, 32, 32, 32, 32, 3], activation=activation, lr=0.5)
    model.train(X, y, epochs=3000, verbose=False)
    results[activation] = model

# Loss curves
axes[0].plot(results['sigmoid'].loss_history, label='Sigmoid', color='#e74c3c', linewidth=2)
axes[0].plot(results['relu'].loss_history, label='ReLU', color='#2ecc71', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Sigmoid vs ReLU in a Deep Network (4 hidden layers)', fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].set_yscale('log')

# Decision boundaries side by side
plt.sca(axes[1])
plot_decision_boundary(results['relu'], X, y, 'ReLU Decision Boundary')

plt.tight_layout()
plt.show()

print(f"Sigmoid final accuracy: {np.mean(results['sigmoid'].predict(X) == y):.1%}")
print(f"ReLU final accuracy:    {np.mean(results['relu'].predict(X) == y):.1%}")

## Exercises

### Exercise A — Implement Momentum
Standard gradient descent can oscillate. Momentum helps by accumulating a velocity vector:
$$v = \beta v - \alpha \nabla L$$
$$\theta = \theta + v$$

Add a `momentum` parameter to the `update()` method and observe the effect on the loss curve.

In [ ]:
# TODO: Implement momentum-based gradient descent
# Hint: add a self.velocities dict in __init__ and update it in the update() method

class NeuralNetworkMomentum(NeuralNetwork):
    def __init__(self, layer_dims, activation='relu', lr=0.01, beta=0.9):
        super().__init__(layer_dims, activation, lr)
        self.beta = beta
        # YOUR CODE HERE: initialize velocity dict
        self.velocities = {}
        for l in range(1, self.L + 1):
            # Initialize velocities to zero for W and b
            pass  # YOUR CODE HERE

    def update(self, grads):
        for l in range(1, self.L + 1):
            # YOUR CODE HERE: implement momentum update
            pass

# Test your implementation
# nn_momentum = NeuralNetworkMomentum([2, 64, 32, 3], lr=0.5, beta=0.9)
# nn_momentum.train(X, y, epochs=5000)
# Compare loss curves with the baseline network

### Exercise B — L2 Regularization
Without regularization, the network can overfit. Modify the `compute_loss()` and `backward()` methods to add L2 regularization:
$$L_{reg} = L + \frac{\lambda}{2N}\sum_l ||W^{[l]}||^2_F$$

Try `lambda = 0.001, 0.01, 0.1` and observe the decision boundary smoothness.

### Exercise C — Deeper Network Experiment  
1. Train a 1-hidden-layer network (2 → 8 → 3). What accuracy can it achieve?
2. Train a 5-hidden-layer network (2 → 64 → 64 → 64 → 64 → 3). Does more depth help?
3. Plot the final decision boundaries side by side.

### Discussion Questions
1. Why does sigmoid suffer from vanishing gradients but ReLU does not?
2. What would happen if we initialized all weights to zero? Try it and explain.
3. The spiral dataset has 3 interleaved classes. How many hidden units are needed minimum? Can you find the minimal network that achieves >95% accuracy?